This project implements an advanced Bird image classification pipeline leveraging transfer learning with a pre-trained ResNet-18 convolutional neural network, fine-tuned on a custom dataset of animal images organized into training, validation, and test splits. Images are preprocessed with resizing to 128×128 pixels and normalization using ImageNet statistics to ensure compatibility with the backbone model. The final fully connected layer of ResNet-18 is replaced to match the number of target classes, and only this layer is trained while the rest of the network remains frozen, significantly reducing training time and overfitting risk. Hyperparameter optimization is performed using Optuna, exploring learning rates, batch sizes, optimizer types (Adam and SGD), dropout rates, and hidden layer sizes to maximize validation accuracy. The model is trained on GPU for efficiency, with batch sizes up to 64 and early stopping based on validation performance. Across approximately 5,000 training images, the optimized model achieves high validation and test accuracy, demonstrating robust generalization. The pipeline also includes a streamlined inference function for real-time prediction on unseen images, making it suitable for practical deployment in animal recognition tasks.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import optuna
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from PIL import Image
from torchvision.transforms.functional import to_pil_image

base_dir = '/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/archive'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'valid')
test_dir = os.path.join(base_dir, 'test')

In [2]:
#  STEP 2: DEFINE TRANSFORMS (Resize, Normalize)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [3]:
#  STEP 3: LOAD DATASETS & DATALOADERS
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)


In [ ]:
# #STEP 4 & 5: MODEL CREATION, TRAINING FUNCTION, AND OPTUNA OBJECTIVE

def train_model(model, optimizer, criterion, train_loader, val_loader, epochs=5):
    best_val_acc = 0
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_correct, train_total = 0, 0
        for images, labels in tqdm(train_loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        train_acc = train_correct / train_total

        # --- Validation ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        best_val_acc = max(best_val_acc, val_acc)
        print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    return best_val_acc

def objective(trial):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Model definition
    model = models.resnet50(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.fc.in_features
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 2)
    hidden_units1 = trial.suggest_int("hidden_units1", 256, 1024, step=128)
    hidden_units2 = trial.suggest_int("hidden_units2", 128, 512, step=64) if num_hidden_layers == 2 else None
    dropout = trial.suggest_float("dropout", 0.2, 0.5)

    if num_hidden_layers == 1:
        model.fc = nn.Sequential(
            nn.Linear(num_ftrs, hidden_units1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units1, num_classes)
        )
    else:
        model.fc = nn.Sequential(
            nn.Linear(num_ftrs, hidden_units1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units1, hidden_units2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units2, num_classes)
        )

    model = model.to(device)

    # Optimizer & learning rate
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0, 1e-3)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.fc.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.fc.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    # Batch size
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Dataloaders for this trial
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()

    # Run training and return best validation accuracy
    return train_model(model, optimizer, criterion, train_loader, val_loader, epochs=5, device=device)

# --- OPTUNA STUDY (optional, comment out if not using) ---
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=10)
# print("Best trial:", study.best_trial.params)

# --- TRAIN FINAL MODEL (without Optuna, if you want to train directly) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Train the model and print train/val accuracy per epoch
train_model(model, optimizer, criterion, train_loader, val_loader, epochs=5)

# --- SAVE FINAL MODEL ---
torch.save(model.state_dict(), '/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/best_model.pth')

# --- TEST ACCURACY ---
def evaluate_accuracy(model, data_loader, device="cpu"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

test_acc = evaluate_accuracy(model, test_loader, device=device)
print(f"Test Accuracy: {test_acc:.4f}")

# --- SIMPLE INFERENCE FUNCTION ---
from PIL import Image

def predict_image(img_path):
    model.eval()
    image = Image.open(img_path).convert('RGB')
    img = val_test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)
        predicted_class = class_names[predicted.item()]
    print(f"Predicted class: {predicted_class}")

# Example usage:
predict_image("/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/archive/images to predict/1.jpg")

Epoch 1: Train Acc = 0.0854, Val Acc = 0.1100


Epoch 2: Train Acc = 0.2304, Val Acc = 0.3300


Epoch 3: Train Acc = 0.4336, Val Acc = 0.4600


Epoch 4: Train Acc = 0.5835, Val Acc = 0.6000


Epoch 5: Train Acc = 0.6802, Val Acc = 0.6800
Test Accuracy: 0.8000
Predicted class: AFRICAN CROWNED CRANE


In [7]:
predict_image("/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/archive/images to predict/6.jpg")

Predicted class: AMERICAN GOLDFINCH
